In [1]:
# Step 1: Import libraries and upload dataset
import pandas as pd
import numpy as np

from google.colab import files
uploaded = files.upload()

Saving heart_statlog_cleveland_hungary_final.csv to heart_statlog_cleveland_hungary_final.csv


In [2]:
filename = list(uploaded.keys())[0]
df = pd.read_csv(filename)

print("Initial shape:", df.shape)
df.head()

Initial shape: (1190, 12)


,age,sex,chest pain type,resting bp s,cholesterol,fasting blood sugar,resting ecg,max heart rate,exercise angina,oldpeak,ST slope,target
0,40,1,2,140,289,0,0,172,0,0.0,1,0
1,49,0,3,160,180,0,0,156,0,1.0,2,1
2,37,1,2,130,283,0,1,98,0,0.0,1,0
3,48,0,4,138,214,0,0,108,1,1.5,2,1
4,54,1,3,150,195,0,0,122,0,0.0,1,0


In [3]:
print("Data types:\n", df.dtypes)
print("\nMissing values per column:\n", df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())
print("\nTarget distribution:\n", df['target'].value_counts())

Data types:
 age                      int64
sex                      int64
chest pain type          int64
resting bp s             int64
cholesterol              int64
fasting blood sugar      int64
resting ecg              int64
max heart rate           int64
exercise angina          int64
oldpeak                float64
ST slope                 int64
target                   int64
dtype: object

Missing values per column:
 age                    0
sex                    0
chest pain type        0
resting bp s           0
cholesterol            0
fasting blood sugar    0
resting ecg            0
max heart rate         0
exercise angina        0
oldpeak                0
ST slope               0
target                 0
dtype: int64

Duplicate rows: 272

Target distribution:
 target
1    629
0    561
Name: count, dtype: int64


In [4]:
df = df.drop_duplicates()
print("Shape after duplicate removal:", df.shape)
print("Duplicate rows remaining:", df.duplicated().sum())
print("Missing values remaining:", df.isnull().sum().sum())
print("\nTarget distribution after dedup:\n", df['target'].value_counts())

Shape after duplicate removal: (918, 12)
Duplicate rows remaining: 0
Missing values remaining: 0

Target distribution after dedup:
 target
1    508
0    410
Name: count, dtype: int64


In [5]:
from sklearn.model_selection import train_test_split

X = df.drop('target', axis=1)
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

print("Training set shape:", X_train.shape)
print("Test set shape:", X_test.shape)
print("\nTraining target distribution:\n", y_train.value_counts(normalize=True))
print("\nTest target distribution:\n", y_test.value_counts(normalize=True))

Training set shape: (734, 11)
Test set shape: (184, 11)

Training target distribution:
 target
1    0.553134
0    0.446866
Name: proportion, dtype: float64

Test target distribution:
 target
1    0.554348
0    0.445652
Name: proportion, dtype: float64


In [6]:
from sklearn.model_selection import train_test_split

X = df.drop('target', axis=1)
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

print("Training set shape:", X_train.shape)
print("Test set shape:", X_test.shape)
print("\nTraining target distribution:\n", y_train.value_counts(normalize=True))
print("\nTest target distribution:\n", y_test.value_counts(normalize=True))

Training set shape: (734, 11)
Test set shape: (184, 11)

Training target distribution:
 target
1    0.553134
0    0.446866
Name: proportion, dtype: float64

Test target distribution:
 target
1    0.554348
0    0.445652
Name: proportion, dtype: float64


In [7]:
from sklearn.impute import SimpleImputer

continuous_cols = ['age', 'resting bp s', 'cholesterol', 'max heart rate', 'oldpeak']
categorical_binary_cols = ['sex', 'chest pain type', 'fasting blood sugar',
                            'resting ecg', 'exercise angina', 'ST slope']

median_imputer = SimpleImputer(strategy='median')
X_train_imputed = X_train.copy()
X_test_imputed = X_test.copy()

X_train_imputed[continuous_cols] = median_imputer.fit_transform(X_train[continuous_cols])
X_test_imputed[continuous_cols] = median_imputer.transform(X_test[continuous_cols])

mode_imputer = SimpleImputer(strategy='most_frequent')
X_train_imputed[categorical_binary_cols] = mode_imputer.fit_transform(X_train[categorical_binary_cols])
X_test_imputed[categorical_binary_cols] = mode_imputer.transform(X_test[categorical_binary_cols])

print("Missing values in X_train after imputation:", X_train_imputed.isnull().sum().sum())
print("Missing values in X_test after imputation:", X_test_imputed.isnull().sum().sum())

Missing values in X_train after imputation: 0
Missing values in X_test after imputation: 0


In [8]:
def get_iqr_bounds(train_col):
    Q1 = train_col.quantile(0.25)
    Q3 = train_col.quantile(0.75)
    IQR = Q3 - Q1
    return Q1 - 1.5 * IQR, Q3 + 1.5 * IQR

bounds = {}
for col in continuous_cols:
    lower, upper = get_iqr_bounds(X_train_imputed[col])
    bounds[col] = (lower, upper)
    n_train = ((X_train_imputed[col] < lower) | (X_train_imputed[col] > upper)).sum()
    n_test = ((X_test_imputed[col] < lower) | (X_test_imputed[col] > upper)).sum()
    print(f"{col}: lower={lower:.2f}, upper={upper:.2f}, outliers_train={n_train}, outliers_test={n_test}")

age: lower=26.00, upper=82.00, outliers_train=0, outliers_test=0
resting bp s: lower=88.50, upper=172.50, outliers_train=20, outliers_test=6
cholesterol: lower=45.38, upper=404.38, outliers_train=140, outliers_test=44
max heart rate: lower=63.50, upper=211.50, outliers_train=2, outliers_test=0
oldpeak: lower=-2.25, upper=3.75, outliers_train=13, outliers_test=3


In [9]:
X_train_capped = X_train_imputed.copy()
X_test_capped = X_test_imputed.copy()

for col in continuous_cols:
    lower, upper = bounds[col]
    X_train_capped[col] = X_train_capped[col].clip(lower, upper)
    X_test_capped[col] = X_test_capped[col].clip(lower, upper)

print("X_train_capped shape:", X_train_capped.shape)
print("X_test_capped shape:", X_test_capped.shape)

X_train_capped shape: (734, 11)
X_test_capped shape: (184, 11)


In [10]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train_capped),
                               columns=X_train_capped.columns, index=X_train_capped.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test_capped),
                              columns=X_test_capped.columns, index=X_test_capped.index)

print("X_train_scaled stats:\n", X_train_scaled.describe().loc[['mean', 'std']])

X_train_scaled stats:
                age           sex  chest pain type  resting bp s   cholesterol  \
mean  2.105491e-16 -9.438408e-17    -1.839280e-16 -1.815079e-16  8.409864e-17   
std   1.000682e+00  1.000682e+00     1.000682e+00  1.000682e+00  1.000682e+00   

      fasting blood sugar   resting ecg  max heart rate  exercise angina  \
mean         1.452063e-17 -1.694073e-17    4.961215e-16    -2.178094e-17   
std          1.000682e+00  1.000682e+00    1.000682e+00     1.000682e+00   

           oldpeak      ST slope  
mean -2.813372e-17  9.680419e-17  
std   1.000682e+00  1.000682e+00  


In [12]:
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold, (train_idx, val_idx) in enumerate(cv.split(X_train_scaled, y_train), 1):
    fold_val_dist = y_train.iloc[val_idx].value_counts(normalize=True).to_dict()
    print(f"Fold {fold}: train_size={len(train_idx)}, val_size={len(val_idx)}, val_class_dist={fold_val_dist}")

Fold 1: train_size=587, val_size=147, val_class_dist={1: 0.5578231292517006, 0: 0.4421768707482993}
Fold 2: train_size=587, val_size=147, val_class_dist={1: 0.5510204081632653, 0: 0.4489795918367347}
Fold 3: train_size=587, val_size=147, val_class_dist={1: 0.5510204081632653, 0: 0.4489795918367347}
Fold 4: train_size=587, val_size=147, val_class_dist={1: 0.5510204081632653, 0: 0.4489795918367347}
Fold 5: train_size=588, val_size=146, val_class_dist={1: 0.5547945205479452, 0: 0.4452054794520548}


In [13]:
!pip install xgboost catboost --quiet

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 1.6 MB/s eta 0:00:00


In [14]:
lr_model = LogisticRegression(random_state=42, max_iter=1000)
lr_model.fit(X_train_scaled, y_train)

rf_model = RandomForestClassifier(random_state=42, n_estimators=200)
rf_model.fit(X_train_scaled, y_train)

xgb_model = XGBClassifier(random_state=42, eval_metric='logloss')
xgb_model.fit(X_train_scaled, y_train)

cat_model = CatBoostClassifier(random_state=42, verbose=0)
cat_model.fit(X_train_scaled, y_train)

print("All four models trained.")

All four models trained.


In [15]:
from sklearn.model_selection import cross_val_score

lr_cv = cross_val_score(lr_model, X_train_scaled, y_train, cv=cv, scoring='accuracy')
rf_cv = cross_val_score(rf_model, X_train_scaled, y_train, cv=cv, scoring='accuracy')
xgb_cv = cross_val_score(xgb_model, X_train_scaled, y_train, cv=cv, scoring='accuracy')
cat_cv = cross_val_score(cat_model, X_train_scaled, y_train, cv=cv, scoring='accuracy')

print(f"Logistic Regression: mean={lr_cv.mean():.4f}, std={lr_cv.std():.4f}")
print(f"Random Forest:       mean={rf_cv.mean():.4f}, std={rf_cv.std():.4f}")
print(f"XGBoost:              mean={xgb_cv.mean():.4f}, std={xgb_cv.std():.4f}")
print(f"CatBoost:              mean={cat_cv.mean():.4f}, std={cat_cv.std():.4f}")

Logistic Regression: mean=0.8379, std=0.0363
Random Forest:       mean=0.8624, std=0.0355
XGBoost:              mean=0.8488, std=0.0211
CatBoost:              mean=0.8679, std=0.0339


In [16]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

def evaluate(model, X_te, name):
    pred = model.predict(X_te)
    proba = model.predict_proba(X_te)[:, 1]
    results = {
        'Accuracy': accuracy_score(y_test, pred),
        'Precision': precision_score(y_test, pred, zero_division=0),
        'Recall': recall_score(y_test, pred, zero_division=0),
        'F1-Score': f1_score(y_test, pred, zero_division=0),
        'ROC-AUC': roc_auc_score(y_test, proba)
    }
    print(f"\n{name}:")
    for k, v in results.items():
        print(f"  {k}: {v:.4f}")
    return results

lr_results = evaluate(lr_model, X_test_scaled, "Logistic Regression")
rf_results = evaluate(rf_model, X_test_scaled, "Random Forest")
xgb_results = evaluate(xgb_model, X_test_scaled, "XGBoost")
cat_results = evaluate(cat_model, X_test_scaled, "CatBoost")


Logistic Regression:
  Accuracy: 0.8859
  Precision: 0.8716
  Recall: 0.9314
  F1-Score: 0.9005
  ROC-AUC: 0.9018

Random Forest:
  Accuracy: 0.8804
  Precision: 0.8774
  Recall: 0.9118
  F1-Score: 0.8942
  ROC-AUC: 0.9336

XGBoost:
  Accuracy: 0.8641
  Precision: 0.8812
  Recall: 0.8725
  F1-Score: 0.8768
  ROC-AUC: 0.9223

CatBoost:
  Accuracy: 0.9076
  Precision: 0.9126
  Recall: 0.9216
  F1-Score: 0.9171
  ROC-AUC: 0.9372


In [17]:
comparison_df = pd.DataFrame({
    'Logistic Regression': lr_results,
    'Random Forest': rf_results,
    'XGBoost': xgb_results,
    'CatBoost': cat_results
}).T

comparison_df = comparison_df.round(4).sort_values(by='F1-Score', ascending=False)
print("Comparative Analysis Table:\n")
comparison_df

Comparative Analysis Table:



,Accuracy,Precision,Recall,F1-Score,ROC-AUC
CatBoost,0.9076,0.9126,0.9216,0.9171,0.9372
Logistic Regression,0.8859,0.8716,0.9314,0.9005,0.9018
Random Forest,0.8804,0.8774,0.9118,0.8942,0.9336
XGBoost,0.8641,0.8812,0.8725,0.8768,0.9223


In [18]:
for name, model in [
    ("Logistic Regression", lr_model),
    ("Random Forest", rf_model),
    ("XGBoost", xgb_model),
    ("CatBoost", cat_model)
]:
    train_acc = accuracy_score(y_train, model.predict(X_train_scaled))
    test_acc = accuracy_score(y_test, model.predict(X_test_scaled))
    print(f"{name}: Train Acc={train_acc:.4f}, Test Acc={test_acc:.4f}, Gap={train_acc-test_acc:.4f}")

Logistic Regression: Train Acc=0.8501, Test Acc=0.8859, Gap=-0.0357
Random Forest: Train Acc=1.0000, Test Acc=0.8804, Gap=0.1196
XGBoost: Train Acc=1.0000, Test Acc=0.8641, Gap=0.1359
CatBoost: Train Acc=0.9741, Test Acc=0.9076, Gap=0.0665


In [19]:
# Regularized Random Forest: limit depth, require more samples per split/leaf
rf_model_reg = RandomForestClassifier(
    random_state=42,
    n_estimators=200,
    max_depth=6,
    min_samples_split=10,
    min_samples_leaf=5,
    max_features='sqrt'
)
rf_model_reg.fit(X_train_scaled, y_train)

# Regularized XGBoost: limit depth, add L1/L2 regularization, lower learning rate
xgb_model_reg = XGBClassifier(
    random_state=42,
    eval_metric='logloss',
    max_depth=4,
    learning_rate=0.05,
    n_estimators=300,
    reg_alpha=1.0,
    reg_lambda=2.0,
    subsample=0.8,
    colsample_bytree=0.8
)
xgb_model_reg.fit(X_train_scaled, y_train)

for name, model in [("Random Forest (reg)", rf_model_reg), ("XGBoost (reg)", xgb_model_reg)]:
    train_acc = accuracy_score(y_train, model.predict(X_train_scaled))
    test_acc = accuracy_score(y_test, model.predict(X_test_scaled))
    print(f"{name}: Train Acc={train_acc:.4f}, Test Acc={test_acc:.4f}, Gap={train_acc-test_acc:.4f}")

Random Forest (reg): Train Acc=0.9074, Test Acc=0.8859, Gap=0.0215
XGBoost (reg): Train Acc=0.9591, Test Acc=0.8913, Gap=0.0678
